# LAB- P8: El Modelo Neoclásico de Crecimiento Exógeno (Solow-Swan)
**Proyecto MACRO-AI-COMP (Convocatoria INNOVA26, UMA / Banco Santander)**
*   **Código de LAB-**: LAB-P8-v1.0
*   **Capítulo de Referencia**: Capítulo 9, *An Introduction to Computational Macroeconomics* (Bongers, Gómez y Torres, Vernon Press, 2019)
*   **Autores**: Equipo Docente MACRO-AI-COMP
*   **Objetivo**: Simular y analizar la dinámica de acumulación de capital en tiempo discreto, el proceso de transición hacia el estado estacionario, los efectos de perturbaciones estructurales (tasa de ahorro, demografía y TFP), y el principio de la Regla de Oro.

---

## Objetivos de Aprendizaje
Al finalizar esta práctica, serás capaz de:
1.  **Comprender** el mecanismo dinámico de acumulación de capital físico per cápita propuesto por Solow (1956) y Swan (1956).
2.  **Identificar** los determinantes de largo plazo de la riqueza y productividad laboral en el estado estacionario (tasa de ahorro, depreciación, crecimiento poblacional y TFP).
3.  **Analizar** la transición dinámica de las variables per cápita (capital, producción, consumo e inversión) tras shocks estructurales.
4.  **Diferenciar** entre el efecto impacto (corto plazo) y el efecto de largo plazo sobre el bienestar tras un incremento del ahorro.
5.  **Explicar** analítica y visualmente el concepto de la **Regla de Oro** y las consecuencias de la ineficiencia dinámica (infra-acumulación vs. sobre-acumulación).
 (Julia)

> **👋 BIENVENIDA A LA PRÁCTICA - LEER ANTES DE EMPEZAR**
> 
> *   **¿Nunca has usado Jupyter?** No te preocupes. Este cuaderno es interactivo. Haz clic en cualquier celda de código y pulsa **`Shift + Enter`** para ejecutarla. Ve de arriba a abajo en orden.
> *   **¿Se ha congelado o sale un asterisco `[*]` eterno?** Ve al menú superior y dale a `Kernel` ➔ `Restart`.
> *   **El objetivo** de esta práctica es que juegues con la economía. Cambia los números del código que representan impuestos, dinero o tecnología, vuelve a ejecutar y mira los gráficos. ¡No puedes romper nada!
>

### 🕹️ GUÍA RÁPIDA PARA DUMMIES - Modelo de Crecimiento de Solow-Swan
*   **¿Qué estamos haciendo aquí?** Analizando por qué crecen los países a largo plazo y cómo acumulan capital (maquinaria).
*   **La Regla de Oro:** Ahorrar más es bueno porque permite comprar más máquinas, pero si ahorras demasiado, no te queda dinero para consumir. Existe una "tasa de ahorro óptima" que maximiza el bienestar.
*   **¡Prueba esto!** Sube la tasa de ahorro y observa cómo la economía transiciona hacia un nivel de riqueza permanente más alto.


In [1]:
# En Google Colab se activarían y descargarían los paquetes necesarios.
# using Pkg; Pkg.activate("."); Pkg.instantiate()


In [2]:
using Pkg
Pkg.activate("../..")

using MacroAIComp
using Plots
import Plots: mm
using LinearAlgebra
using Interact
using BenchmarkTools


  Activating project at `C:\Users\AntonioRC\Desktop\PIE`


WebIO._IJuliaInit()

## 1. El Modelo Neoclásico de Crecimiento (Solow-Swan)

El modelo de Solow-Swan es el bloque fundamental de la teoría moderna del crecimiento económico. A diferencia de los modelos de equilibrio general dinámico con agentes optimizadores (como Ramsey o DGE básico), este modelo asume que **la tasa de ahorro es exógena y constante**. 

### 1.1 Estructura Matemática

El modelo se define en variables per cápita (por trabajador) para aislar el tamaño de la economía y capturar el bienestar individual:
*   **Población (Trabajo):** Crece a una tasa constante $n$:
    $$L_{t} = L_{t-1}(1 + n)$$
*   **Función de Producción Intensiva (Cobb-Douglas):**
    $$y_t = A_t k_t^\alpha$$
    donde $y_t \equiv Y_t/L_t$ es la producción per cápita, $k_t \equiv K_t/L_t$ es el stock de capital per cápita, $A_t$ es la TFP, y $\alpha$ es la elasticidad capital-trabajo.
*   **Ahorro e Inversión:** Una fracción constante $s$ de la producción se destina a la inversión bruta:
    $$i_t = s y_t$$
*   **Consumo per cápita:** Lo que no se ahorra se consume:
    $$c_t = (1 - s)y_t$$

### 1.2 Dinámica del Capital y Estado Estacionario
La acumulación de capital per cápita sigue la siguiente ecuación en diferencias de primer orden:
$$k_{t+1} = \frac{(1 - \delta)k_t + s A_t k_t^\alpha}{1 + n}$$

El estado estacionario ($\bar{k}$) ocurre cuando el capital por trabajador permanece constante ($\Delta k_t = 0$), lo que implica que la inversión por trabajador cubre exactamente la depreciación física y el crecimiento de la población (depreciación efectiva):
$$s A \bar{k}^\alpha = (\delta + n)\bar{k}$$

Despejando el capital per cápita estacionario:
$$\bar{k} = \left( \frac{\delta + n}{s A} \right)^{\frac{1}{\alpha - 1}}$$

Una vez obtenido $\bar{k}$, las demás variables se calculan de manera directa:
$$\bar{y} = A \bar{k}^\alpha, \quad \bar{c} = (1 - s)\bar{y}, \quad \bar{i} = s\bar{y}$$


In [3]:
params_base = default_calibration(SolowSwanParameters)
ss = compute_solow_steady_state(params_base)

println("VALORES DE ESTADO ESTACIONARIO:")
println("  Capital por trabajador (k*)   : ", round(ss["k"], digits=4))
println("  Producción por trabajador (y*): ", round(ss["y"], digits=4))
println("  Consumo por trabajador (c*)   : ", round(ss["c"], digits=4))
println("  Inversión por trabajador (i*) : ", round(ss["i"], digits=4))


VALORES DE ESTADO ESTACIONARIO:
  Capital por trabajador (k*)   : 4.0946
  Producción por trabajador (y*): 1.6379
  Consumo por trabajador (c*)   : 1.3103
  Inversión por trabajador (i*) : 0.3276


## 2. Simulación Interactiva: Transición Dinámica y Shocks

Supongamos que la economía parte de su estado estacionario base con una tasa de ahorro del $20\%$ ($s_0 = 0.20$), depreciación del $6\%$ ($\delta = 0.06$) y crecimiento poblacional del $2\%$ ($n_0 = 0.02$). En el período $t = 5$, la economía sufre una perturbación permanente en su tasa de ahorro (por ejemplo, sube al $25\%$).

Con el siguiente panel interactivo, puedes mover la tasa de ahorro final ($s_1$), el crecimiento poblacional final ($n_1$), y la TFP final ($A_1$) para analizar cómo responde la economía período a período.


In [4]:
# Simulación interactiva: Transición Dinámica de Solow
@manipulate for s_final in 0.10:0.01:0.50, n_final in 0.00:0.005:0.05, A_final in 0.5:0.1:2.0
    
    params_init = default_calibration(SolowSwanParameters)
    ss_init = compute_solow_steady_state(params_init)
    k0 = ss_init["k"]
    T_sim = 100
    
    # Send shocks at t=10
    s_path = fill(params_init.s, T_sim)
    s_path[11:end] .= s_final
    
    n_path = fill(params_init.n, T_sim)
    n_path[11:end] .= n_final
    
    A_path = fill(params_init.A, T_sim)
    A_path[11:end] .= A_final
    
    res = simulate_solow_swan(params_init, k0, s_path, n_path, A_path, T_sim)
    
    params_fin = SolowSwanParameters(params_init.alpha, params_init.delta, s_final, n_final, A_final)
    ss_fin = compute_solow_steady_state(params_fin)
    
    t_axis = 0:(T_sim - 1)
    
    p1 = plot(t_axis, res["k"], color=:blue, lw=2.5, label="Capital (k)")
    hline!([ss_init["k"]], color=:gray, ls=:dot, label="")
    hline!([ss_fin["k"]], color=:black, ls=:dash, label="k* Final")
    title!("Capital por trabajador")
    xlabel!("Tiempo")
    
    p2 = plot(t_axis, res["y"], color=:purple, lw=2.5, label="Renta (y)")
    hline!([ss_init["y"]], color=:gray, ls=:dot, label="")
    hline!([ss_fin["y"]], color=:black, ls=:dash, label="y* Final")
    title!("Renta per cápita")
    xlabel!("Tiempo")
    
    p3 = plot(t_axis, res["c"], color=:forestgreen, lw=2.5, label="Consumo (c)")
    hline!([ss_init["c"]], color=:gray, ls=:dot, label="")
    hline!([ss_fin["c"]], color=:black, ls=:dash, label="c* Final")
    title!("Consumo per cápita")
    xlabel!("Tiempo")
    
    plot(p1, p2, p3, layout=(1,3), size=(1100, 350), 
         plot_title="Ajuste hacia el Nuevo Estado Estacionario", top_margin=10mm)
end


Node{WebIO.DOM}(WebIO.DOM(:html, :div), Any[Node{WebIO.DOM}(WebIO.DOM(:html, :div), Any[Scope(Node{WebIO.DOM}(WebIO.DOM(:html, :div), Any[Node{WebIO.DOM}(WebIO.DOM(:html, :div), Any[Node{WebIO.DOM}(WebIO.DOM(:html, :label), Any["s_final"], Dict{Symbol, Any}(:className => "interact ", :style => Dict{Any, Any}(:padding => "5px 10px 0px 10px")))], Dict{Symbol, Any}(:className => "interact-flex-row-left")), Node{WebIO.DOM}(WebIO.DOM(:html, :div), Any[Node{WebIO.DOM}(WebIO.DOM(:html, :input), Any[], Dict{Symbol, Any}(:max => 41, :min => 1, :attributes => Dict{Any, Any}(:type => "range", Symbol("data-bind") => "numericValue: index, valueUpdate: 'input', event: {change: function (){this.changes(this.changes()+1)}}", "orient" => "horizontal"), :step => 1, :className => "slider slider is-fullwidth", :style => Dict{Any, Any}()))], Dict{Symbol, Any}(:className => "interact-flex-row-center")), Node{WebIO.DOM}(WebIO.DOM(:html, :div), Any[Node{WebIO.DOM}(WebIO.DOM(:html, :p), Any[], Dict{Symbol, Any}(:attributes => Dict("data-bind" => "text: formatted_val")))], Dict{Symbol, Any}(:className => "interact-flex-row-right"))], Dict{Symbol, Any}(:className => "interact-flex-row interact-widget")), Dict{String, Tuple{AbstractObservable, Union{Nothing, Bool}}}("changes" => (Observable(0), nothing), "index" => (Observable{Any}(21), nothing)), Set{String}(), nothing, Asset[Asset("js", "knockout", "C:\\Users\\AntonioRC\\.julia\\packages\\Knockout\\HReiN\\src\\..\\assets\\knockout.js"), Asset("js", "knockout_punches", "C:\\Users\\AntonioRC\\.julia\\packages\\Knockout\\HReiN\\src\\..\\assets\\knockout_punches.js"), Asset("js", nothing, "C:\\Users\\AntonioRC\\.julia\\packages\\InteractBase\\8TTmI\\src\\..\\assets\\all.js"), Asset("css", nothing, "C:\\Users\\AntonioRC\\.julia\\packages\\InteractBase\\8TTmI\\src\\..\\assets\\style.css"), Asset("css", nothing, "C:\\Users\\AntonioRC\\.julia\\packages\\Interact\\PENUy\\src\\..\\assets\\bulma_confined.min.css")], Dict{Any, Any}("changes" => Any[WebIO.JSString("(function (val){return (val!=this.model[\"changes\"]()) ? (this.valueFromJulia[\"changes\"]=true, this.model[\"changes\"](val)) : undefined})")], "index" => Any[WebIO.JSString("(function (val){return (val!=this.model[\"index\"]()) ? (this.valueFromJulia[\"index\"]=true, this.model[\"index\"](val)) : undefined})")]), WebIO.ConnectionPool(Channel{Any}(32), Set{AbstractConnection}(), Base.GenericCondition(ReentrantLock())), WebIO.JSString[WebIO.JSString("function () {\n    var handler = (function (ko, koPunches) {\n    ko.punches.enableAll();\n    ko.bindingHandlers.numericValue = {\n        init: function(element, valueAccessor, allBindings, data, context) {\n            var stringified = ko.observable(ko.unwrap(valueAccessor()));\n            stringified.subscribe(function(value) {\n                var val = parseFloat(value);\n                if (!isNaN(val)) {\n                    valueAccessor()(val);\n                }\n            });\n            valueAccessor().subscribe(function(value) {\n                var str = JSON.stringify(value);\n                if ((str == \"0\") && ([\"-0\", \"-0.\"].indexOf(stringified()) >= 0))\n                     return;\n                 if ([\"null\", \"\"].indexOf(str) >= 0)\n                     return;\n                stringified(str);\n            });\n            ko.applyBindingsToNode(\n                element,\n                {\n                    value: stringified,\n                    valueUpdate: allBindings.get('valueUpdate'),\n                },\n                context,\n            );\n        }\n    };\n    var json_data = {\"formatted_vals\":[\"0.1\",\"0.11\",\"0.12\",\"0.13\",\"0.14\",\"0.15\",\"0.16\",\"0.17\",\"0.18\",\"0.19\",\"0.2\",\"0.21\",\"0.22\",\"0.23\",\"0.24\",\"0.25\",\"0.26\",\"0.27\",\"0.28\",\"0.29\",\"0.3\",\"0.31\",\"0.32\",\"0.33\",\"0.34\",\"0.35\",\"0.36\",\"0.37\",\"0.38\",\"0.39\",\"0.4\",\"0.41\",\"0.42\",\"0.43\",\"0.44\",\"0.45\",\"0.46\",\"0.47\",\"0.48\",\"0.49\",\"

## 3. La Regla de Oro (Golden Rule) de Acumulación de Capital

Dado que un mayor ahorro implica mayor capital y producción de largo plazo, ¿debe una economía ahorrar tanto como sea posible? La respuesta es **no**. 

El objetivo último de la actividad económica es el **consumo** y el bienestar de los hogares, no la acumulación de capital por sí misma. Si ahorramos una fracción muy alta (cercana al $100\%$), la producción será inmensa, pero casi toda se reinvertirá para compensar la depreciación del gigantesco stock de capital, dejando un consumo cercano a cero. Si ahorramos una fracción muy baja, el capital se agotará y la producción será insignificante, resultando también en un consumo residual.

### 3.1 Derivación Analítica
La tasa de ahorro de la **Regla de Oro** ($s^{gold}$) es aquella que maximiza el consumo de estado estacionario:
$$\max_{s} \bar{c}(s) = \bar{y} - (\delta + n)\bar{k}$$
Sustituyendo $\bar{y} = A \bar{k}^\alpha$:
$$\max_{k} \bar{c} = A \bar{k}^\alpha - (\delta + n)\bar{k}$$
La condición de primer orden con respecto a $\bar{k}$ es:
$$\frac{d\bar{c}}{d\bar{k}} = \alpha A \bar{k}^{\alpha-1} - (\delta + n) = 0 \implies \alpha \frac{\bar{y}}{\bar{k}} = \delta + n$$

Multiplicando ambos lados por $\bar{k}/\bar{y}$ y sabiendo que en estado estacionario $s \frac{\bar{y}}{\bar{k}} = \delta + n$, llegamos a:
$$s^{gold} = \alpha$$

*   **Ineficiencia Dinámica:** Si $s > \alpha$, la economía está sobre-acumulando capital. Podría aumentar el consumo tanto hoy como a largo plazo simplemente reduciendo su tasa de ahorro.
*   **Bajo-acumulación:** Si $s < \alpha$, la economía está consumiendo demasiado a corto plazo, de modo que un incremento del ahorro aumentaría el consumo en el futuro.


In [5]:
# Demostración Visual de la Regla de Oro
@manipulate for s_current in 0.05:0.01:0.60
    
    params = default_calibration(SolowSwanParameters)
    alpha_val = params.alpha
    delta_val = params.delta
    n_val = params.n
    A_val = params.A
    
    # Calcular la tasa de ahorro de la regla de oro (s_gold = alpha en Solow simple)
    s_gold = alpha_val
    k_gold = (s_gold * A_val / (delta_val + n_val))^(1 / (1 - alpha_val))
    c_gold = A_val * k_gold^alpha_val - (delta_val + n_val) * k_gold
    
    # Estado estacionario actual
    k_curr = (s_current * A_val / (delta_val + n_val))^(1 / (1 - alpha_val))
    y_curr = A_val * k_curr^alpha_val
    i_curr = s_current * y_curr
    c_curr = y_curr - i_curr
    
    # Crear malla de capitales para graficar
    k_vals = range(0.1, 50.0, length=200)
    y_vals = A_val .* k_vals .^ alpha_val
    inv_vals = s_current .* y_vals
    req_inv = (delta_val + n_val) .* k_vals
    
    # Gráfica
    p1 = plot(k_vals, y_vals, color=:purple, lw=3, label="Producción f(k)")
    plot!(k_vals, inv_vals, color=:blue, lw=2.5, label="Ahorro/Inv. s_current")
    plot!(k_vals, req_inv, color=:red, lw=2.5, label="Inv. Requerida (n+δ)k")
    
    # Marcar el punto actual
    vline!([k_curr], color=:gray, ls=:dot, lw=2, label="k* Actual")
    scatter!([k_curr], [y_curr], color=:purple, markersize=6, label="")
    scatter!([k_curr], [req_inv[argmin(abs.(k_vals .- k_curr))]], color=:red, markersize=6, label="")
    
    # Llenar área de consumo actual
    plot!( [k_curr, k_curr], [i_curr, y_curr], color=:forestgreen, lw=5, label="Consumo actual: $(round(c_curr, digits=2))" )
    
    # Marcar Golden Rule
    vline!([k_gold], color=:orange, ls=:dash, lw=2, label="k* Golden Rule")
    scatter!([k_gold], [A_val * k_gold^alpha_val], color=:orange, markersize=8, marker=:star, label="Max Consumo")
    
    title!("Regla de Oro en Solow-Swan")
    xlabel!("Capital por trabajador (k)")
    ylabel!("Renta, Inversión")
    
    plot(p1, size=(700, 450), plot_title="Comparativa Consumo vs Golden Rule", top_margin=10mm)
end


Node{WebIO.DOM}(WebIO.DOM(:html, :div), Any[Node{WebIO.DOM}(WebIO.DOM(:html, :div), Any[Scope(Node{WebIO.DOM}(WebIO.DOM(:html, :div), Any[Node{WebIO.DOM}(WebIO.DOM(:html, :div), Any[Node{WebIO.DOM}(WebIO.DOM(:html, :label), Any["s_current"], Dict{Symbol, Any}(:className => "interact ", :style => Dict{Any, Any}(:padding => "5px 10px 0px 10px")))], Dict{Symbol, Any}(:className => "interact-flex-row-left")), Node{WebIO.DOM}(WebIO.DOM(:html, :div), Any[Node{WebIO.DOM}(WebIO.DOM(:html, :input), Any[], Dict{Symbol, Any}(:max => 56, :min => 1, :attributes => Dict{Any, Any}(:type => "range", Symbol("data-bind") => "numericValue: index, valueUpdate: 'input', event: {change: function (){this.changes(this.changes()+1)}}", "orient" => "horizontal"), :step => 1, :className => "slider slider is-fullwidth", :style => Dict{Any, Any}()))], Dict{Symbol, Any}(:className => "interact-flex-row-center")), Node{WebIO.DOM}(WebIO.DOM(:html, :div), Any[Node{WebIO.DOM}(WebIO.DOM(:html, :p), Any[], Dict{Symbol, Any}(:attributes => Dict("data-bind" => "text: formatted_val")))], Dict{Symbol, Any}(:className => "interact-flex-row-right"))], Dict{Symbol, Any}(:className => "interact-flex-row interact-widget")), Dict{String, Tuple{AbstractObservable, Union{Nothing, Bool}}}("changes" => (Observable(0), nothing), "index" => (Observable{Any}(28), nothing)), Set{String}(), nothing, Asset[Asset("js", "knockout", "C:\\Users\\AntonioRC\\.julia\\packages\\Knockout\\HReiN\\src\\..\\assets\\knockout.js"), Asset("js", "knockout_punches", "C:\\Users\\AntonioRC\\.julia\\packages\\Knockout\\HReiN\\src\\..\\assets\\knockout_punches.js"), Asset("js", nothing, "C:\\Users\\AntonioRC\\.julia\\packages\\InteractBase\\8TTmI\\src\\..\\assets\\all.js"), Asset("css", nothing, "C:\\Users\\AntonioRC\\.julia\\packages\\InteractBase\\8TTmI\\src\\..\\assets\\style.css"), Asset("css", nothing, "C:\\Users\\AntonioRC\\.julia\\packages\\Interact\\PENUy\\src\\..\\assets\\bulma_confined.min.css")], Dict{Any, Any}("changes" => Any[WebIO.JSString("(function (val){return (val!=this.model[\"changes\"]()) ? (this.valueFromJulia[\"changes\"]=true, this.model[\"changes\"](val)) : undefined})")], "index" => Any[WebIO.JSString("(function (val){return (val!=this.model[\"index\"]()) ? (this.valueFromJulia[\"index\"]=true, this.model[\"index\"](val)) : undefined})")]), WebIO.ConnectionPool(Channel{Any}(32), Set{AbstractConnection}(), Base.GenericCondition(ReentrantLock())), WebIO.JSString[WebIO.JSString("function () {\n    var handler = (function (ko, koPunches) {\n    ko.punches.enableAll();\n    ko.bindingHandlers.numericValue = {\n        init: function(element, valueAccessor, allBindings, data, context) {\n            var stringified = ko.observable(ko.unwrap(valueAccessor()));\n            stringified.subscribe(function(value) {\n                var val = parseFloat(value);\n                if (!isNaN(val)) {\n                    valueAccessor()(val);\n                }\n            });\n            valueAccessor().subscribe(function(value) {\n                var str = JSON.stringify(value);\n                if ((str == \"0\") && ([\"-0\", \"-0.\"].indexOf(stringified()) >= 0))\n                     return;\n                 if ([\"null\", \"\"].indexOf(str) >= 0)\n                     return;\n                stringified(str);\n            });\n            ko.applyBindingsToNode(\n                element,\n                {\n                    value: stringified,\n                    valueUpdate: allBindings.get('valueUpdate'),\n                },\n                context,\n            );\n        }\n    };\n    var json_data = {\"formatted_vals\":[\"0.05\",\"0.06\",\"0.07\",\"0.08\",\"0.09\",\"0.1\",\"0.11\",\"0.12\",\"0.13\",\"0.14\",\"0.15\",\"0.16\",\"0.17\",\"0.18\",\"0.19\",\"0.2\",\"0.21\",\"0.22\",\"0.23\",\"0.24\",\"0.25\",\"0.26\",\"0.27\",\"0.28\",\"0.29\",\"0.3\",\"0.31\",\"0.32\",\"0.33\",\"0.34\",\"0.35\",\"0.36\",\"0.37\",\"0.38\",\"0.39\",\"0.4\",\"0.41\",\"0.42\",\"0.43\",\"0.44\",

## 4. Cuaderno de Bitácora (Actividades para el Alumno)

Responde a las siguientes cuestiones tras interactuar con las simulaciones del modelo de crecimiento de Solow-Swan:

1.  **Mecanismo de Transición y Sacrificio de Consumo**:
    *   Establece una tasa de ahorro final de $s_1 = 0.25$ (partiendo de $s_0 = 0.20$). Observa la gráfica de consumo per cápita. ¿Qué ocurre exactamente en el período $t=5$ (momento del shock)? ¿Por qué cae el consumo instantáneamente si la producción se mantiene igual?
    *   Explica cómo evoluciona el consumo a partir de $t=6$. ¿Por qué se recupera y acaba superando al nivel inicial? Relaciona esto con el concepto de "acumulación de capital".
2.  **La Regla de Oro y la Ineficiencia Dinámica**:
    *   Utiliza el panel de la Regla de Oro. Establece la tasa de ahorro actual en $s = 0.50$. ¿Qué le ocurre al consumo estacionario comparado con el de la Regla de Oro ($s=0.35$)? ¿Por qué se dice que una economía con $s = 0.50$ es dinámicamente ineficiente? ¿Cómo podría esa economía mejorar el bienestar tanto hoy como en el futuro?
3.  **Transición Demográfica y TFP**:
    *   En la simulación de shocks (Sección 2), devuelve el ahorro al $20\%$ y reduce la tasa de crecimiento demográfico de $n_0 = 0.02$ a $n_1 = 0.00$. Describe la dinámica del capital per cápita y de la producción per cápita. ¿Por qué una población constante ($n=0$) eleva la riqueza por trabajador a largo plazo?
    *   Ahora, aplica un incremento permanente de la TFP del $5\%$ ($A = 1.05$). ¿Qué ocurre con la tasa de crecimiento del PIB per cápita en el año del shock y a largo plazo? Explica por qué, en ausencia de crecimiento continuo de la TFP ($g_A = 0$), el crecimiento de la renta per cápita a largo plazo acaba siendo cero en este modelo.


## 5. Buenas Prácticas Aplicadas en este Laboratorio
1.  **Modularización del Modelo**: Las ecuaciones y simulaciones del modelo de Solow-Swan no están dispersas, sino importadas del módulo de biblioteca [`growth.py`](file:///c:/Users/AntonioRC/Desktop/PIE/src/macroaicomp/models/growth.py), aislando la visualización del backend computacional.
2.  **Diseño Paramétrico Limpio**: La calibración y los shocks se definen mediante objetos `SolowSwanParameters` y vectores ordenados de tiempo, facilitando la escalabilidad del simulador.
3.  **Higiene del Repositorio**: El cuaderno está integrado en el flujo de trabajo de Git con `nbstripout` y controles linter en el hook de `pre-commit`, previniendo que los gráficos pesados contaminen el repositorio y asegurando un arranque limpio.


## 6. Benchmark de Rendimiento (Fase III)
Evaluamos la velocidad de simulación usando `BenchmarkTools.jl`.

In [6]:
# Benchmark simulation
T_bench = 100
s_path = fill(0.20, T_bench); n_path = fill(0.015, T_bench); A_path = fill(1.0, T_bench)
@btime simulate_solow_swan($params_base, $ss["k"], $s_path, $n_path, $A_path, $T_bench)


  3.925 μs (14 allocations: 4.97 KiB)


Dict{String, Vector{Float64}} with 9 entries:
  "Y"  => [1.63785, 1.62921, 1.62107, 1.61341, 1.60618, 1.59939, 1.59299, 1.586…
  "i"  => [0.327571, 0.325843, 0.324215, 0.322681, 0.321237, 0.319877, 0.318597…
  "I"  => [0.327571, 0.325843, 0.324215, 0.322681, 0.321237, 0.319877, 0.318597…
  "c"  => [1.31028, 1.30337, 1.29686, 1.29072, 1.28495, 1.27951, 1.27439, 1.269…
  "gy" => [0.0, -0.00527581, -0.00499673, -0.00473004, -0.00447545, -0.00423264…
  "C"  => [1.31028, 1.30337, 1.29686, 1.29072, 1.28495, 1.27951, 1.27439, 1.269…
  "k"  => [4.09464, 4.03322, 3.9759, 3.92241, 3.87246, 3.82581, 3.78224, 3.7415…
  "K"  => [4.09464, 4.03322, 3.9759, 3.92241, 3.87246, 3.82581, 3.78224, 3.7415…
  "y"  => [1.63785, 1.62921, 1.62107, 1.61341, 1.60618, 1.59939, 1.59299, 1.586…